In [34]:
import pandas as pd

import gc
import os
import torch

from translate_df_col import translate_df_column
from text_classification import (
    TextClassifierTrainConfig,
    prepare_dataframe,
    stratified_split,
)

# Setup inicial

In [2]:
# 1) borra referencias grandes que sigan vivas
for name in [
    "model", "optimizer", "scheduler",
    "train_loader", "val_loader",
    "train_ds", "val_ds",
    "batch", "outputs", "loss", "logits", "preds"
]:
    if name in globals():
        del globals()[name]

# 2) limpia RAM de Python
gc.collect()

# 3) intenta sincronizar y vaciar caché CUDA
if torch.cuda.is_available():
    try:
        torch.cuda.synchronize()
    except Exception:
        pass

    try:
        torch.cuda.empty_cache()
    except Exception as e:
        print("empty_cache falló:", repr(e))

    try:
        torch.cuda.ipc_collect()
    except Exception as e:
        print("ipc_collect falló:", repr(e))

# División train-val

In [ ]:
PATH_TO_TRAIN_VAL_DF = 'path/to/train&val_df'

df_train_val = pd.read_excel(
    PATH_TO_TRAIN_VAL_DF,
    )

df_train_val_translated, errs_train_val = translate_df_column(df_train_val, col = "comentario")

In [5]:
config = TextClassifierTrainConfig(
    model_name="answerdotai/ModernBERT-base",  # o "answerdotai/ModernBERT-base"
    text_col="comentario_en",
    label_col = "categoria",
    valid_labels = (1, 2, 3, 4, 5, 6, 7, 8, 9),
    class_names = {1: 'Comunicación y Presencia',
                   2: 'Salud',
                   3: 'Asesoría preventiva',
                   4: 'Capacitaciones',
                   5: 'Plataforma',
                   6: 'SEL',
                   7: 'Calificación',
                   8: 'Requerimiento',
                   9: 'Otros'},
    val_size=0.12,
)

# 1) preparar y limpiar
df_train_val_ready, code2idx, idx2code, id2label, label2id = prepare_dataframe(
    df_train_val_translated,
    config,
)

print("Tamaño dataset listo:", df_train_val_ready.shape)
print(df_train_val_ready[config.label_col].value_counts().sort_index())

# 2) split estratificado
train_df, val_df = stratified_split(df_train_val_ready, config)

Tamaño dataset listo: (6500, 6)
categoria
1    2198
2    1006
3     613
4     240
5     283
6     146
7     130
8    1176
9     708
Name: count, dtype: int64


In [9]:
train_df

,encuesta_id,audio_id,comentario,categoria,comentario_en,label_idx
0,51418,51418.0,PORQUE HEMOS TENIDO BUENA PREOCUPACIÓN DE ELLO...,1,BECAUSE WE HAVE HAD GOOD CONCERN FROM THEM TOW...,0
1,138432,138432.0,PORQUE ALGUNOS REQUERIMIENTOS NO SON ACEPTADOS...,8,BECAUSE SOME REQUIREMENTS ARE NOT ACCEPTED DUE...,7
2,19322,2322525.0,PORQUE LA ACHS SE SUPONE QUE APOYA A LAS AEMPR...,1,BECAUSE THE ACHS IS SUPPOSED TO SUPPORT COMPAN...,0
3,6016,46113296.0,"POR QUE TAMPOCO, LOS SERVICIOS SON BUENOS PERO...",9,"BECAUSE EITHER, THE SERVICES ARE GOOD BUT NOT ...",8
4,12378,1404508.0,PORQUE SIEMPRE ME HAN DADO SOLUCIONES. HORAS ...,8,BECAUSE THEY HAVE ALWAYS GAVE ME SOLUTIONS. F...,7
...,...,...,...,...,...,...
5715,158,54141251.0,ES QUE SE HAN SOLICITADI VARIAS GESTIONES PREV...,3,SEVERAL PREVENTIVE MEASURES HAVE BEEN REQUESTE...,2
5716,10075,63818933.0,HE QUEDADO SATISFECHO CON TODOS LOS SERVICIOS ...,8,I HAVE BEEN SATISFIED WITH ALL THE SERVICES TH...,7
5717,22270,3295056.0,PORQUE LAS EMPRESAS PEQUEÑAS NO EXISTIMOS PARA...,1,BECAUSE SMALL BUSINESSES DO NOT EXIST FOR THE ...,0
5718,12611,1448058.0,PORQUE LO HACE MUY BIEN. MUY BUENA. NADA.,9,BECAUSE HE DOES IT VERY WELL. VERY GOOD. NOT...,8


In [10]:
val_df

,encuesta_id,audio_id,comentario,categoria,comentario_en,label_idx
0,12398,1481967.0,PORQUE NO ESTOY MUY CONFORME. SI COMO SE PUEDE...,1,"BECAUSE I AM NOT VERY HAPPY. YES, AS YOU CAN S...",0
1,12945,1461835.0,LO QUE PASA QUE SIEMPRE TENGO QUE ESTAR LLAMAN...,1,"WHAT HAPPENS THAT I ALWAYS HAVE TO BE CALLING,...",0
2,145026,145026.0,POR LA ATENCION LA VERDAD MIS TRABAJADORES HA...,1,FOR THE ATTENTION THE TRUTH MY WORKERS HAVE CO...,0
3,14205,1712673.0,PORQUE ULTIMAMENTE NO ES MUCHA LA RELACION QUE...,1,BECAUSE LATESTLY THERE IS NOT MUCH RELATIONSHI...,0
4,12422,1404132.0,PORQUE DE MOMENTOS PROBLEMAS QUE HEMOS TENIDO ...,2,BECAUSE OF MOMENTS WE HAVE HAD PROBLEMS WITH S...,1
...,...,...,...,...,...,...
775,21042,2907136.0,PORQUE YO NOSE NO TENGO MUCHA INFORMACION COMO...,2,BECAUSE I DON'T KNOW I DON'T HAVE MUCH INFORMA...,1
776,12916,1470652.0,PORQUE A MI COMO EMPLEADOR NO SIENTO QUE ME A...,4,BECAUSE TO ME AS AN EMPLOYER I DON'T FEEL THAT...,3
777,234976,234976.0,POR LA EXPERIENCIA QUE HEMOS TENIDO TRABAJANDO...,1,FOR THE EXPERIENCE WE HAVE HAD WORKING WITH YOU.,0
778,12996,1514652.0,POR UN TEMA DE COMUNICACIÓN FALTA DE COMUNICA...,1,DUE TO A COMMUNICATION ISSUE LACK OF COMMUNICA...,0


In [ ]:
OUTPUT_PATH_TRAIN_DF = 'output/path/train_df'
OUTPUT_PATH_VAL_DF = 'output/path/val_df'

train_df.to_excel(OUTPUT_PATH_TRAIN_DF, index= False)
val_df.to_excel(OUTPUT_PATH_VAL_DF, index= False)

# Back Translate

In [ ]:
from text_data_augment import augment_df_with_backtranslation

In [ ]:
train_df = pd.read_excel(OUTPUT_PATH_TRAIN_DF)
train_df

,encuesta_id,audio_id,comentario,categoria,comentario_en,label_idx
0,51418,51418.0,PORQUE HEMOS TENIDO BUENA PREOCUPACIÓN DE ELLO...,1,BECAUSE WE HAVE HAD GOOD CONCERN FROM THEM TOW...,0
1,138432,138432.0,PORQUE ALGUNOS REQUERIMIENTOS NO SON ACEPTADOS...,8,BECAUSE SOME REQUIREMENTS ARE NOT ACCEPTED DUE...,7
2,19322,2322525.0,PORQUE LA ACHS SE SUPONE QUE APOYA A LAS AEMPR...,1,BECAUSE THE ACHS IS SUPPOSED TO SUPPORT COMPAN...,0
3,6016,46113296.0,"POR QUE TAMPOCO, LOS SERVICIOS SON BUENOS PERO...",9,"BECAUSE EITHER, THE SERVICES ARE GOOD BUT NOT ...",8
4,12378,1404508.0,PORQUE SIEMPRE ME HAN DADO SOLUCIONES. HORAS ...,8,BECAUSE THEY HAVE ALWAYS GAVE ME SOLUTIONS. F...,7
...,...,...,...,...,...,...
5715,158,54141251.0,ES QUE SE HAN SOLICITADI VARIAS GESTIONES PREV...,3,SEVERAL PREVENTIVE MEASURES HAVE BEEN REQUESTE...,2
5716,10075,63818933.0,HE QUEDADO SATISFECHO CON TODOS LOS SERVICIOS ...,8,I HAVE BEEN SATISFIED WITH ALL THE SERVICES TH...,7
5717,22270,3295056.0,PORQUE LAS EMPRESAS PEQUEÑAS NO EXISTIMOS PARA...,1,BECAUSE SMALL BUSINESSES DO NOT EXIST FOR THE ...,0
5718,12611,1448058.0,PORQUE LO HACE MUY BIEN. MUY BUENA. NADA.,9,BECAUSE HE DOES IT VERY WELL. VERY GOOD. NOT...,8


In [ ]:
print(train_df['categoria'].value_counts().sort_index())

categoria
1    1934
2     885
3     540
4     211
5     249
6     129
7     114
8    1035
9     623
Name: count, dtype: int64


In [ ]:
# El criterio de ¿qué clases aumentan cuánto?, depende de la cantidad proporcional de cada clase

train_df_augment_2 = train_df[train_df['categoria'].isin([2])].copy()
train_df_augment_3 = train_df[train_df['categoria'].isin([3])].copy()
train_df_augment_5 = train_df[train_df['categoria'].isin([4,5])].copy()
train_df_augment_8 = train_df[train_df['categoria'].isin([6,7])].copy()

In [68]:
multi_lenguages_paths_2 = [[["fr"],], 
                           [["de"],]]
multi_lenguages_paths_3 = [[["fr"],], 
                           [["de"],],
                           [["pt"],]]
multi_lenguages_paths_5 = [[["fr"],], 
                           [["de"],],
                           [["pt"],],
                           [["it"],],
                           [["fr","de"],],]
multi_lenguages_paths_8 = [[["fr"],], 
                           [["de"],],
                           [["pt"],],
                           [["it"],],
                           [["fr","it"],],
                           [["pt","it"],],
                           [["fr","de"],],
                           [["pt","de"],],
                           ]


def multi_back_translate_df_augment(train_df_, multi_lenguages_paths, col = 'comentario_en'):
    augmented_dfs = []
    errors_list = []
    for language_paths in multi_lenguages_paths:
        df_bt_aug, errors = augment_df_with_backtranslation(
            df=train_df_,
            text_col=col,
            original_lang="en",
            language_paths=language_paths,
            keep_original=True,
        )
        augmented_dfs.append(df_bt_aug)
        errors_list.append(errors)
    return augmented_dfs, errors_list

In [ ]:
# múltiples aumentos de un df:
augmented_dfs_8, errors_list_8 = multi_back_translate_df_augment(train_df_augment_8, multi_lenguages_paths_8, col = 'comentario_en')

Translating en->fr:   0%|          | 0/243 [00:00<?, ?it/s]

Translating de->en: 100%|██████████| 243/243 [00:03<00:00, 80.66it/s]


In [ ]:
df_augmented_8_back_translate = pd.concat(augmented_dfs_8)
df_augmented_8_back_translate.drop_duplicates(subset='comentario_en', inplace=True)

In [ ]:
# Si se usan múltiples listas de secuencias (multi_lenguages_paths_N), hay que repetir el proceso para dichas listas, luego concatenar y limpiar todo antes de guardar
# df_augmented_8_back_translate.to_excel("save_path_back_translate_data_aug", index=False)

# Synonym Replacement

In [ ]:
from text_data_augment import augment_df_with_synonym_replacement

In [ ]:
train_df = pd.read_excel(OUTPUT_PATH_TRAIN_DF)
train_df

,encuesta_id,audio_id,comentario,categoria,comentario_en,label_idx
0,51418,51418.0,PORQUE HEMOS TENIDO BUENA PREOCUPACIÓN DE ELLO...,1,BECAUSE WE HAVE HAD GOOD CONCERN FROM THEM TOW...,0
1,138432,138432.0,PORQUE ALGUNOS REQUERIMIENTOS NO SON ACEPTADOS...,8,BECAUSE SOME REQUIREMENTS ARE NOT ACCEPTED DUE...,7
2,19322,2322525.0,PORQUE LA ACHS SE SUPONE QUE APOYA A LAS AEMPR...,1,BECAUSE THE ACHS IS SUPPOSED TO SUPPORT COMPAN...,0
3,6016,46113296.0,"POR QUE TAMPOCO, LOS SERVICIOS SON BUENOS PERO...",9,"BECAUSE EITHER, THE SERVICES ARE GOOD BUT NOT ...",8
4,12378,1404508.0,PORQUE SIEMPRE ME HAN DADO SOLUCIONES. HORAS ...,8,BECAUSE THEY HAVE ALWAYS GAVE ME SOLUTIONS. F...,7
...,...,...,...,...,...,...
5715,158,54141251.0,ES QUE SE HAN SOLICITADI VARIAS GESTIONES PREV...,3,SEVERAL PREVENTIVE MEASURES HAVE BEEN REQUESTE...,2
5716,10075,63818933.0,HE QUEDADO SATISFECHO CON TODOS LOS SERVICIOS ...,8,I HAVE BEEN SATISFIED WITH ALL THE SERVICES TH...,7
5717,22270,3295056.0,PORQUE LAS EMPRESAS PEQUEÑAS NO EXISTIMOS PARA...,1,BECAUSE SMALL BUSINESSES DO NOT EXIST FOR THE ...,0
5718,12611,1448058.0,PORQUE LO HACE MUY BIEN. MUY BUENA. NADA.,9,BECAUSE HE DOES IT VERY WELL. VERY GOOD. NOT...,8


In [16]:
print(train_df['categoria'].value_counts().sort_index())

categoria
1    1934
2     885
3     540
4     211
5     249
6     129
7     114
8    1035
9     623
Name: count, dtype: int64


In [ ]:
####################################################
# Para 1 y 8 no se hará aumento por sinónimos 
# Para código [2,9,3] se usará N_to_generate=1, 
# Para códigos [4,5,6,7] N_to_generate=2, 

# la lógica anterior queda a criterio del usuario
####################################################

train_df_N_to_gen_1 = train_df[train_df['categoria'].isin([2,3,9])].copy()
train_df_N_to_gen_3 = train_df[train_df['categoria'].isin([4,5,6,7])].copy()

train_df_code239 = augment_df_with_synonym_replacement(
    df=train_df_N_to_gen_1,
    text_col="comentario_en",
    ratio_to_replace=0.40,
    N_to_generate=1,
    keep_original=True,
    random_state=42,
)


train_df_code4567 = augment_df_with_synonym_replacement(
    df=train_df_N_to_gen_3,
    text_col="comentario_en",
    ratio_to_replace=0.40,
    N_to_generate=3,
    keep_original=True,
    random_state=42,
)


In [20]:
train_df_sr_aug_d = pd.concat([
    train_df[train_df['categoria'].isin([1])].copy()[['comentario_en', 'categoria', 'label_idx']].copy(),
    train_df_code239[['comentario_en', 'categoria', 'label_idx']].copy(),
    train_df_code4567[['comentario_en', 'categoria', 'label_idx']],
])

train_df_sr_aug_d = train_df_sr_aug_d.sample(frac=1).reset_index(drop=True)

In [24]:
print(train_df_sr_aug_d['categoria'].value_counts().sort_index())
train_df_sr_aug_d.drop_duplicates(subset='comentario_en', inplace = True)

categoria
1    1934
2    1769
3    1078
4     841
5     996
6     516
7     455
9    1243
Name: count, dtype: int64


In [ ]:
# train_df_sr_aug_d.to_excel("save_path_synonym_replacement_data_aug", index= False)

# Contextual augmentation: replacements & insertions

In [25]:
from text_data_augment import augment_df_with_contextual_augmentation

In [ ]:
train_df = pd.read_excel(OUTPUT_PATH_TRAIN_DF)
train_df

,comentario_en,categoria,label_idx
0,BECAUSE I HAVE RARELY ATTENDED THE ACHS AND I ...,2,1
1,BECAUSE THE TRUTH IS COMPLICATED TO HANDLE WIT...,5,4
2,BECAUSE THE TRUTH IS THAT REGARDING PREVENTIVE...,6,5
3,"BASICALLY BECAUSE OF THE HELP OF THE EXPERTS, ...",3,2
4,BECAUSE WE HAD AN ACCIDENT WITH A WORKER AND T...,1,0
...,...,...,...
8827,BECAUSE THEY Be Unspoiled FROM WHAT I Hold Mee...,6,5
8828,"BECAUSE I HAVE NEVER SEEN THE ACHS, WE CONTACT...",1,0
8829,"BECAUSE OF THE TIME THEY ARE, NEVER, ONLY ONCE...",1,0
8830,"THERE ARE THINGS TO IMPROVE, THE EVALUATION OF...",7,6


In [27]:
train_df['categoria'].value_counts()

categoria
1    1934
2    1769
9    1243
3    1078
5     996
4     841
6     516
7     455
Name: count, dtype: int64

In [ ]:
####################################################
# Para [1,2,9] se usará N_to_generate=1 
# Para código [3,4,5] se usará N_to_generate=3, 
# Para códigos [6,7] N_to_generate=5, 

# la lógica anterior queda a criterio del usuario
####################################################

train_df_N_to_gen_1 = train_df[train_df['categoria'].isin([1,2,9])].copy()
train_df_N_to_gen_3 = train_df[train_df['categoria'].isin([3,4,5])].copy()
train_df_N_to_gen_5 = train_df[train_df['categoria'].isin([6,7])].copy()

# Contextual replacements
train_df_ctx_replace_aug_N_to_gen_1 = augment_df_with_contextual_augmentation(
    df=train_df_N_to_gen_1,
    text_col="comentario_en",
    ratio_to_replace=0.25,
    N_to_generate=1,
    action="substitute",
    model_path="distilbert-base-uncased",
    keep_original=True,
)

train_df_ctx_replace_aug_N_to_gen_3 = augment_df_with_contextual_augmentation(
    df=train_df_N_to_gen_3,
    text_col="comentario_en",
    ratio_to_replace=0.25,
    N_to_generate=3,
    action="substitute",
    model_path="distilbert-base-uncased",
    keep_original=True,
)

train_df_ctx_replace_aug_N_to_gen_5 = augment_df_with_contextual_augmentation(
    df=train_df_N_to_gen_5,
    text_col="comentario_en",
    ratio_to_replace=0.25,
    N_to_generate=5,
    action="substitute",
    model_path="distilbert-base-uncased",
    keep_original=True,
)


# Contextual inserts
train_df_ctx_insert_aug_N_to_gen_1 = augment_df_with_contextual_augmentation(
    df=train_df_N_to_gen_1,
    text_col="comentario_en",
    ratio_to_replace=0.25,
    N_to_generate=1,
    action="insert",
    model_path="distilbert-base-uncased",
    keep_original=True,
)

train_df_ctx_insert_aug_N_to_gen_3 = augment_df_with_contextual_augmentation(
    df=train_df_N_to_gen_3,
    text_col="comentario_en",
    ratio_to_replace=0.25,
    N_to_generate=3,
    action="insert",
    model_path="distilbert-base-uncased",
    keep_original=True,
)

train_df_ctx_insert_aug_N_to_gen_5 = augment_df_with_contextual_augmentation(
    df=train_df_N_to_gen_5,
    text_col="comentario_en",
    ratio_to_replace=0.25,
    N_to_generate=5,
    action="insert",
    model_path="distilbert-base-uncased",
    keep_original=True,
)


The following layers were not sharded: vocab_projector.weight, distilbert.transformer.layer.*.attention.v_lin.weight, distilbert.embeddings.position_embeddings.weight, distilbert.transformer.layer.*.output_layer_norm.bias, distilbert.transformer.layer.*.attention.q_lin.bias, distilbert.transformer.layer.*.attention.k_lin.bias, distilbert.transformer.layer.*.attention.q_lin.weight, distilbert.transformer.layer.*.attention.out_lin.weight, vocab_transform.bias, distilbert.transformer.layer.*.sa_layer_norm.bias, distilbert.embeddings.LayerNorm.bias, distilbert.transformer.layer.*.ffn.lin*.bias, distilbert.transformer.layer.*.ffn.lin*.weight, vocab_layer_norm.weight, vocab_layer_norm.bias, distilbert.embeddings.LayerNorm.weight, distilbert.embeddings.word_embeddings.weight, distilbert.transformer.layer.*.sa_layer_norm.weight, distilbert.transformer.layer.*.output_layer_norm.weight, distilbert.transformer.layer.*.attention.out_lin.bias, vocab_transform.weight, vocab_projector.bias, distilber

In [29]:
train_df_ctx_ins_repl_aug_d = pd.concat([
    train_df_ctx_insert_aug_N_to_gen_1[['comentario_en', 'categoria', 'label_idx']],
    train_df_ctx_insert_aug_N_to_gen_3[['comentario_en', 'categoria', 'label_idx']],
    train_df_ctx_insert_aug_N_to_gen_5[['comentario_en', 'categoria', 'label_idx']],
    train_df_ctx_replace_aug_N_to_gen_1[['comentario_en', 'categoria', 'label_idx']],
    train_df_ctx_replace_aug_N_to_gen_3[['comentario_en', 'categoria', 'label_idx']],
    train_df_ctx_replace_aug_N_to_gen_5[['comentario_en', 'categoria', 'label_idx']],
])
train_df_ctx_ins_repl_aug_d.drop_duplicates(subset='comentario_en', inplace=True)
train_df_ctx_ins_repl_aug_d = train_df_ctx_ins_repl_aug_d.sample(frac=1).reset_index(drop=True)

In [33]:
train_df

,comentario_en,categoria,label_idx
0,BECAUSE I HAVE RARELY ATTENDED THE ACHS AND I ...,2,1
1,BECAUSE THE TRUTH IS COMPLICATED TO HANDLE WIT...,5,4
2,BECAUSE THE TRUTH IS THAT REGARDING PREVENTIVE...,6,5
3,"BASICALLY BECAUSE OF THE HELP OF THE EXPERTS, ...",3,2
4,BECAUSE WE HAD AN ACCIDENT WITH A WORKER AND T...,1,0
...,...,...,...
8827,BECAUSE THEY Be Unspoiled FROM WHAT I Hold Mee...,6,5
8828,"BECAUSE I HAVE NEVER SEEN THE ACHS, WE CONTACT...",1,0
8829,"BECAUSE OF THE TIME THEY ARE, NEVER, ONLY ONCE...",1,0
8830,"THERE ARE THINGS TO IMPROVE, THE EVALUATION OF...",7,6


In [30]:
print(train_df_ctx_ins_repl_aug_d['categoria'].value_counts().sort_index())
train_df_ctx_ins_repl_aug_d

categoria
1    5802
2    5307
3    7546
4    5887
5    6972
6    5676
7    5005
9    3729
Name: count, dtype: int64


,comentario_en,categoria,label_idx
0,but because we had estimable service i have no...,2,1
1,i have just identified technological accidents...,2,1
2,information technology ' s that i could not gi...,4,3
3,because with these new regulations government ...,3,2
4,yesterday i remember it is difficult to ask fo...,4,3
...,...,...,...
45919,because the achs platform infrastructure is no...,5,4
45920,"because they don ' t reply on request, nor hav...",4,3
45921,a percept of the servicing in ecumenical. for ...,2,1
45922,because the honourable robert william service ...,4,3


In [ ]:
train_df_ctx_ins_repl_aug_d.to_excel("save_path_ctx_ins_repl_data_aug", index= False)